# «Фауст»: обучение своих моделей разбора фраз (v4)

Две модели на одних и тех же данных (~19 тысяч фраз «фраза → команда»):

1. **Qwen2.5-1.5B + LoRA (v4)** — дообучение готовой модели: учатся только «заплатки» (~1% чисел).
2. **«Фауст-мини»** — своя маленькая нейросеть (≈0,5 млн чисел), обучается **с нуля**.

**Перед запуском** (справа, панель *Settings*):
- *Accelerator* — **GPU T4 x2** (или P100);
- *Internet* — **On** (скачать Qwen с Hugging Face и библиотеки);
- *Add Input* — датасет с архивом `faust-ml-v4.zip`.

Лучше запускать через **Save Version → Save & Run All**: тогда блокнот доработает в фоне,
даже если закрыть вкладку, а результат появится во вкладке *Output* (`faust-v4-results.zip`).
Обучение Qwen — порядка часа-полутора, мини-модель — минуты.

In [ ]:
# 1. Какая видеокарта досталась
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Код и данные из датасета — в рабочую папку (там можно писать)
import glob, os, shutil
src = glob.glob('/kaggle/input/**/faust/jarvis-ml/common.py', recursive=True)
if not src:   # архив не распакован Kaggle'ом — распакуем сами
    import zipfile
    z = glob.glob('/kaggle/input/**/faust-ml-v4.zip', recursive=True)[0]
    zipfile.ZipFile(z).extractall('/kaggle/working')
else:
    shutil.copytree(os.path.dirname(os.path.dirname(src[0])), '/kaggle/working/faust', dirs_exist_ok=True)
os.chdir('/kaggle/working/faust/jarvis-ml')
print(os.getcwd()); print(os.listdir('.')); print(os.listdir('data'))

In [ ]:
# 3. Библиотеки (те же основные версии, что на ПК: transformers 4.x, peft)
!pip install -q "transformers>=4.46,<5" "peft>=0.13" "accelerate>=1.0"
# В образе Kaggle стоит torchao 0.10, а peft отказывается работать с torchao старше 0.16
# (первый запуск 13.11 упал на get_peft_model). Нам torchao не нужна — убираем.
!pip uninstall -y -q torchao
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 4. Базовая модель Qwen2.5-1.5B-Instruct (открытая, ~3 ГБ) — с Hugging Face
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', local_dir='models/Qwen2.5-1.5B-Instruct')

In [ ]:
# 5. Дообучение Qwen (LoRA). Точность выбирается сама: на T4/P100 — fp16 с масштабированием.
#    Контрольная точка каждые 100 шагов; если сессия оборвётся — тот же запуск с --resume продолжит.
!python -u train.py --base models/Qwen2.5-1.5B-Instruct --name qwen15-v4 --epochs 2 --batch 8 --accum 2 --val-sample 200 --checkpoint-every 100 --precision auto 2>&1 | grep -v Warning
# Команды через «!» не останавливают блокнот при ошибке — проверяем результат сами,
# иначе он пойдёт дальше впустую (так и было в первом запуске).
assert os.path.exists('runs/qwen15-v4/adapter/adapter_model.safetensors'), 'обучение Qwen не удалось — см. вывод выше'

In [ ]:
# 6. Экзамены Qwen v4 сразу здесь (напрямую, без Ollama): А, Б, «Новое» и dev (для порога уверенности)
for s in ['test', 'test_b', 'new', 'dev']:
    !python eval_local.py --name qwen15-v4 --set {s} 2>&1 | tail -14

In [ ]:
# 7. «Фауст-мини» с нуля — на видеокарте это минуты
!python -u mini/train_mini.py --name mini-v1-kaggle --gpu --epochs 40
!python mini/eval_mini.py --name mini-v1-kaggle 2>&1 | grep -E '^==|^ 0.9|^ 0.0'

In [ ]:
# 8. Всё нужное — в один архив: заплатки LoRA, журналы обучения, мини-модель, отчёты экзаменов.
#    Базовую модель не кладём (её скачаем на ПК сами), контрольные точки тоже.
import zipfile
out = '/kaggle/working/faust-v4-results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for root in ['runs/qwen15-v4', 'runs/mini-v1-kaggle', '../jarvis-relay/eval/results']:
        for dirpath, _, files in os.walk(root):
            if 'checkpoint' in dirpath:
                continue
            for f in files:
                p = os.path.join(dirpath, f)
                z.write(p, arcname=os.path.relpath(p, '/kaggle/working/faust'))
print(out, round(os.path.getsize(out) / 2**20, 1), 'МБ')